# SpaceX Data Collection
**Purpose:** retrieve the course-provided SpaceX launch JSON and inspect its structure before extracting analysis fields.

This notebook uses the IBM Skills Network course dataset. Run the cells in order. If the hosted dataset is temporarily unavailable, check the URL before substituting another source.

In [1]:
import requests
import pandas as pd
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
raw = response.json()
print("Records retrieved:", len(raw))
print("First record keys:", list(raw[0].keys()) if raw else "No records")

Records retrieved: 107
First record keys: ['fairings', 'links', 'static_fire_date_utc', 'static_fire_date_unix', 'tbd', 'net', 'window', 'rocket', 'success', 'details', 'crew', 'ships', 'capsules', 'payloads', 'launchpad', 'auto_update', 'failures', 'flight_number', 'name', 'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming', 'cores', 'id']


## Inspect nested fields
The source contains nested launch, payload and core fields. Inspecting a record first helps avoid assuming that all launches have the same nested values.

In [2]:
display(pd.json_normalize(raw).head())

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,...,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],...,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-fa...,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Successful first stage burn and transition to ...,[],[],...,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-roc...,https://en.wikipedia.org/wiki/DemoSat,NaN
2,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision betwe...,[],[],...,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1...,https://en.wikipedia.org/wiki/Trailblazer_(sat...,NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,Ratsat was carried to orbit on the first succe...,[],[],...,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=dLQ2tZEH6G0,dLQ2tZEH6G0,https://en.wikipedia.org/wiki/Ratsat,https://en.wikipedia.org/wiki/Ratsat,NaN
4,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,True,NaN,[],[],...,NaN,NaN,[],[],http://www.spacex.com/press/2012/12/19/spacexs...,https://www.youtube.com/watch?v=yTaIDooc8Og,yTaIDooc8Og,http://www.spacex.com/news/2013/02/12/falcon-1...,https://en.wikipedia.org/wiki/RazakSAT,NaN


## Create a launch-level table
This extraction is defensive: it handles empty lists and missing fields. The course dataset can differ from the live SpaceX API schema, so review the preview and missing-value summary.

In [3]:
def first_item(value):
    return value[0] if isinstance(value, list) and value else {}

rows = []
for item in raw:
    payload = first_item(item.get("payloads"))
    core = first_item(item.get("cores"))
    rows.append({
        "FlightNumber": item.get("flight_number"),
        "Date": item.get("date_utc"),
        "Name": item.get("name"),
        "PayloadMass": payload.get("mass_kg") if isinstance(payload, dict) else None,
        "Orbit": payload.get("orbit") if isinstance(payload, dict) else None,
        "Outcome": core.get("landing_success") if isinstance(core, dict) else None,
        "Flights": core.get("flight") if isinstance(core, dict) else None,
        "GridFins": core.get("gridfins") if isinstance(core, dict) else None,
        "Reused": core.get("reused") if isinstance(core, dict) else None,
        "LandingPad": core.get("landpad") if isinstance(core, dict) else None,
        "BoosterSerial": core.get("core") if isinstance(core, dict) else None
    })
launch_df = pd.DataFrame(rows)
display(launch_df.head())
print("Shape:", launch_df.shape)
display(launch_df.isna().sum().to_frame("missing_values"))

,FlightNumber,Date,Name,PayloadMass,Orbit,Outcome,Flights,GridFins,Reused,LandingPad,BoosterSerial
0,1,2006-03-24T22:30:00.000Z,FalconSat,None,None,None,1,False,False,NaN,5e9e289df35918033d3b2623
1,2,2007-03-21T01:10:00.000Z,DemoSat,None,None,None,1,False,False,NaN,5e9e289ef35918416a3b2624
2,3,2008-08-03T03:34:00.000Z,Trailblazer,None,None,None,1,False,False,NaN,5e9e289ef3591814873b2625
3,4,2008-09-28T23:15:00.000Z,RatSat,None,None,None,1,False,False,NaN,5e9e289ef3591855dc3b2626
4,5,2009-07-13T03:35:00.000Z,RazakSat,None,None,None,1,False,False,NaN,5e9e289ef359184f103b2627


Shape: (107, 11)


,missing_values
FlightNumber,0
Date,0
Name,0
PayloadMass,107
Orbit,107
Outcome,31
Flights,0
GridFins,0
Reused,0
LandingPad,36


In [4]:
launch_df.to_csv("spacex_api_launches.csv", index=False)
print("Saved: spacex_api_launches.csv")

Saved: spacex_api_launches.csv
